In [5]:
import pandas as pd
import numpy as np
import networkx as nx

from networkx.algorithms.community import louvain_communities

In [6]:
trust = pd.read_csv(
    "../epinions/epinions/user_rating.txt",
    sep=r"\s+",
    header=None,
    names=["source", "target", "value", "creation"]
)

mc = pd.read_csv(
    "../epinions/epinions/mc.txt",
    sep="|",
    header=None,
    names=["content_id", "author_id", "subject_id"]
)

ratings = pd.read_csv(
    "../epinions/epinions/rating.txt",
    sep=r"\s+",
    header=None,
    names=[
        "object_id",
        "member_id",
        "rating",
        "status",
        "creation",
        "last_modified",
        "type",
        "vertical_id"
    ]
)

In [7]:
trust["relationship"] = np.where(
    trust["value"] == 1,
    "Trust",
    "Distrust"
)

In [8]:
trust.head(10)

,source,target,value,creation,relationship
0,3287060356,232085,-1,2001/01/10,Distrust
1,3288305540,709420,1,2001/01/10,Trust
2,3290337156,204418,-1,2001/01/10,Distrust
3,3294138244,269243,-1,2001/01/10,Distrust
4,3294138244,170692484,-1,2001/01/10,Distrust
5,3296759684,347967364,1,2001/01/10,Trust
6,167022468,707034,1,2001/01/10,Trust
7,168267652,214139,1,2001/01/10,Trust
8,169447300,338102,1,2001/01/10,Trust
9,169447300,382733,1,2001/01/10,Trust


In [9]:
interaction = ratings.merge(
    mc,
    left_on="object_id",
    right_on="content_id",
    how="inner"
)

In [10]:
interaction = interaction.rename(columns={
    "member_id": "source",
    "author_id": "target"
})

In [11]:
interaction = interaction[
    interaction["source"] != interaction["target"]
]

In [12]:
interaction["relationship"] = "ReviewRating"

In [13]:
interaction = interaction[
    ["source", "target", "rating", "relationship"]
]

In [14]:
interaction.tail(10)

,source,target,rating,relationship
13668094,505314,342187,5,ReviewRating
13668095,234283,342187,5,ReviewRating
13668096,3264581508,342187,5,ReviewRating
13668097,225095,342187,5,ReviewRating
13668098,256565,342187,5,ReviewRating
13668099,311238,342187,5,ReviewRating
13668100,210412,342187,5,ReviewRating
13668101,351471,342187,5,ReviewRating
13668102,394639,342187,5,ReviewRating
13668103,200338,342187,4,ReviewRating


In [15]:
users = pd.unique(
    pd.concat([
        trust["source"],
        trust["target"]
    ])
)

users = pd.Series(users, name="user_id")

print("Number of users:", len(users))

Number of users: 131828


In [16]:
# Feature 1 : activity

# Activity of a user is defined by the ratings given by him/her and ratings received.
ratings_given = (
    interaction.groupby("source")
    .size()
    .rename("ratings_given")
)

reviews_written = (
    interaction.groupby("target")
    .size()
    .rename("reviews_written")
)

activity = pd.concat(
    [ratings_given, reviews_written],
    axis=1
).fillna(0)

activity["Activity"] = (
    activity["ratings_given"] +
    activity["reviews_written"]
)

activity = activity[["Activity"]]

In [17]:
# Feature 2 : Popularity

# Popularity of a user = number of users who trust u
# only +ve trust edges

positive_trust = trust[trust["value"] == 1]

popularity = (
    positive_trust.groupby("target")
    .size()
    .rename("Popularity")
    .to_frame()
)

In [18]:
popularity

,Popularity
target,
199781,37
199784,7
199786,2
199795,3
199820,1
...,...
83785387908,1
83799805828,1
83869405060,1


In [19]:
# Feature 3 : Influence

#Influence(u) = PageRank(u)

G_trust = nx.DiGraph()

G_trust.add_nodes_from(users)

G_trust.add_edges_from(
    positive_trust[["source", "target"]]
    .itertuples(index=False, name=None)
)

print(G_trust.number_of_nodes())
print(G_trust.number_of_edges())

131828
717667


In [20]:
pagerank = nx.pagerank(
    G_trust,
    alpha=0.85
)

In [21]:
influence = pd.Series(
    pagerank,
    name="Influence"
).to_frame()

influence.index.name = "user_id"

In [22]:
# Feature 4 : Community Participation

G_undirected = G_trust.to_undirected()
communities = louvain_communities(
    G_undirected,
    seed=42
)

print("Number of communities:", len(communities))


Number of communities: 24934


In [23]:
community_map = {}

for community_id, community in enumerate(communities):
    for user in community:
        community_map[user] = community_id

community_df = pd.Series(
    community_map,
    name="community_id"
).to_frame()

community_df.index.name = "user_id"

In [24]:
# Community participation = number of user's trust connections within their community/ total trust connections of user
# does this user mainly interact with members of their own community ?

community_edges = positive_trust.copy()

community_edges["source_community"] = (
    community_edges["source"].map(community_map)
)

community_edges["target_community"] = (
    community_edges["target"].map(community_map)
)

community_edges["within_community"] = (
    community_edges["source_community"] ==
    community_edges["target_community"]
)

internal_edges = (
    community_edges[
        community_edges["within_community"]
    ]
    .groupby("source")
    .size()
    .rename("internal_connections")
)

total_connections = (
    positive_trust.groupby("source")
    .size()
    .rename("total_connections")
)

community_participation = pd.concat(
    [internal_edges, total_connections],
    axis=1
).fillna(0)

community_participation["CommunityParticipation"] = np.where(
    community_participation["total_connections"] > 0,
    community_participation["internal_connections"] /
    community_participation["total_connections"],
    0
)

community_participation = community_participation[
    ["CommunityParticipation"]
]

In [25]:
community_participation.head() #between values 0 and 1

,CommunityParticipation
source,
199781,0.5
199784,1.0
199786,1.0
199788,1.0
199797,1.0


In [26]:
# Feature 5 : Reciprocity

# Reciprocity(u) = reciprocated outgoing trust relationships / total outgoing trust relationships
trust_pairs = set(
    zip(
        positive_trust["source"],
        positive_trust["target"]
    )
)

positive_trust["is_reciprocal"] = [
    (target, source) in trust_pairs
    for source, target
    in zip(
        positive_trust["source"],
        positive_trust["target"]
    )
]

reciprocity = (
    positive_trust
    .groupby("source")["is_reciprocal"]
    .mean()
    .rename("Reciprocity")
    .to_frame()
)

#Example : User has 10 outgoing trust links
# 6 have a return trust link
# reciprocity = 6/10 = 0.6

C:\Users\admin\AppData\Local\Temp\ipykernel_27440\4048527519.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_trust["is_reciprocal"] = [


In [27]:
# Feature 6 : Reputation

# Defined by average rating received by user u
# 1 ≤ Reputation(u) ≤ 5

reputation = (
    interaction
    .groupby("target")["rating"]
    .mean()
    .rename("Reputation")
    .to_frame()
)


In [28]:
# Combine all 6 features

features = pd.DataFrame(index=users)

features.index.name = "user_id"

features = features.join(activity)
features = features.join(popularity)
features = features.join(influence)
features = features.join(community_participation)
features = features.join(reciprocity)
features = features.join(reputation)

In [29]:
features.head()

,Activity,Popularity,Influence,CommunityParticipation,Reciprocity,Reputation
user_id,,,,,,
3287060356,NaN,NaN,0.000002,NaN,NaN,NaN
3288305540,1.0,1.0,0.000002,1.000000,0.000000,NaN
3290337156,21.0,1.0,0.000003,0.777778,0.111111,NaN
3294138244,7.0,NaN,0.000002,NaN,NaN,NaN
3296759684,3.0,NaN,0.000002,1.000000,0.000000,NaN


In [30]:
features.shape

(131828, 6)

In [31]:
# handle missing values

features = features.fillna(0)

In [32]:
features.shape

(131828, 6)

In [33]:
features.head()

,Activity,Popularity,Influence,CommunityParticipation,Reciprocity,Reputation
user_id,,,,,,
3287060356,0.0,0.0,0.000002,0.000000,0.000000,0.0
3288305540,1.0,1.0,0.000002,1.000000,0.000000,0.0
3290337156,21.0,1.0,0.000003,0.777778,0.111111,0.0
3294138244,7.0,0.0,0.000002,0.000000,0.000000,0.0
3296759684,3.0,0.0,0.000002,1.000000,0.000000,0.0


In [34]:
features.to_csv(
    "epinions_user_features_full.csv"
)

In [35]:
# Normalize the six features : required for probability distribution

feature_cols = [
    "Activity",
    "Popularity",
    "Influence",
    "CommunityParticipation",
    "Reciprocity",
    "Reputation"
]

X = features[feature_cols].copy()

In [49]:
correlation_matrix = features[feature_cols].corr(method="pearson")

display(correlation_matrix.round(4))

,Activity,Popularity,Influence,CommunityParticipation,Reciprocity,Reputation
Activity,1.0000,0.8676,0.7994,0.0143,0.1360,0.1447
Popularity,0.8676,1.0000,0.9443,0.0213,0.1855,0.1850
Influence,0.7994,0.9443,1.0000,0.0217,0.1952,0.1634
CommunityParticipation,0.0143,0.0213,0.0217,1.0000,0.2953,-0.1424
Reciprocity,0.1360,0.1855,0.1952,0.2953,1.0000,0.2745
Reputation,0.1447,0.1850,0.1634,-0.1424,0.2745,1.0000


In [36]:
# Create probability distributions

P = X.div(
    X.sum(axis=0),
    axis=1 )

print(P.sum())

Activity                  1.0
Popularity                1.0
Influence                 1.0
CommunityParticipation    1.0
Reciprocity               1.0
Reputation                1.0
dtype: float64


In [37]:
# Probability-distribution negation

N = len(P)
Q = (1 - P) / (N - 1)

In [38]:
# Calculate the 6 Euclidean distances

distances = np.sqrt(
    ((P - Q) ** 2).sum(axis=0)
)

In [39]:
distance_df = pd.DataFrame({
    "Feature": feature_cols,
    "EuclideanDistance": distances.values
})

distance_df

,Feature,EuclideanDistance
0,Activity,0.023195
1,Popularity,0.016360
2,Influence,0.013106
3,CommunityParticipation,0.002153
4,Reciprocity,0.005449
5,Reputation,0.002759


In [40]:
# Calculate the feature weights

weights = distances / distances.sum()
weight_df = pd.DataFrame({
    "Feature": feature_cols,
    "Distance": distances.values,
    "Weight": weights.values
})


In [41]:
weight_df

,Feature,Distance,Weight
0,Activity,0.023195,0.368037
1,Popularity,0.016360,0.259593
2,Influence,0.013106,0.207955
3,CommunityParticipation,0.002153,0.034167
4,Reciprocity,0.005449,0.086464
5,Reputation,0.002759,0.043783


In [42]:
print("Weight sum:", weights.sum())

Weight sum: 0.9999999999999999


In [43]:
# Calculate each user's trustworthiness score after min max normalization of every feature value

X_norm = X.copy()

for col in feature_cols:
    min_val = X_norm[col].min()
    max_val = X_norm[col].max()

    if max_val > min_val:
        X_norm[col] = (
            (X_norm[col] - min_val) /
            (max_val - min_val)
        )
    else:
        X_norm[col] = 0

trust_score = (
    X_norm * weights
).sum(axis=1)

In [44]:
top_user = 210284

# Global min and max for each feature
min_max = pd.DataFrame({
    "Min": X[feature_cols].min(),
    "Max": X[feature_cols].max()
})

print("Global Min-Max:")
display(min_max)

# Normalized values for top user
top_normalized = X_norm.loc[top_user]

# Contribution of each feature
contributions = top_normalized * weights

top_user_calculation = pd.DataFrame({
    "Raw Value": X.loc[top_user],
    "Normalized Value": top_normalized,
    "Weight": weights,
    "Contribution": contributions
})

display(top_user_calculation)

print("Final Trustworthiness:", contributions.sum())

Global Min-Max:


,Min,Max
Activity,0.000000,196497.000000
Popularity,0.000000,3338.000000
Influence,0.000002,0.003559
CommunityParticipation,0.000000,1.000000
Reciprocity,0.000000,1.000000
Reputation,0.000000,6.000000


,Raw Value,Normalized Value,Weight,Contribution
Activity,196497.000000,1.000000,0.368037,0.368037
Popularity,3338.000000,1.000000,0.259593,0.259593
Influence,0.003559,1.000000,0.207955,0.207955
CommunityParticipation,0.295455,0.295455,0.034167,0.010095
Reciprocity,0.795455,0.795455,0.086464,0.068778
Reputation,4.887801,0.814634,0.043783,0.035667


Final Trustworthiness: 0.9501257534633588


In [45]:
features["Trustworthiness"] = trust_score

In [46]:
results = features.sort_values(
    "Trustworthiness",
    ascending=False
)

# Sort users by trustworthiness

results.head(20)
# top 20 trusted users


,Activity,Popularity,Influence,CommunityParticipation,Reciprocity,Reputation,Trustworthiness
user_id,,,,,,,
210284,196497.0,3338.0,0.003559,0.295455,0.795455,4.887801,0.950126
204418,89564.0,1222.0,0.001123,0.704819,0.947791,4.974044,0.470679
335034,88452.0,989.0,0.001083,0.864865,0.967568,4.991540,0.455408
200118,36488.0,1536.0,0.002367,0.599045,0.837709,4.980196,0.455331
295491,76144.0,1246.0,0.001239,0.788732,0.645540,5.016934,0.431215
205639,69898.0,1673.0,0.001446,0.376812,0.381643,4.821288,0.426538
346777,58682.0,1190.0,0.001454,0.888655,0.825630,4.978321,0.425416
200338,75230.0,1252.0,0.000998,0.694737,0.789474,4.875572,0.424108
243427,58778.0,1139.0,0.001141,0.665188,0.935698,4.942800,0.404992


In [47]:
results.to_csv(
    "epinions_user_trust_scores.csv"
)

In [48]:
weight_df.to_csv(
    "epinions_feature_weights.csv",
    index=False
)